# Optimisation & Benchmarks — Partie 2

**Objectifs** : comparer implementation naive (`iterrows`) vs vectorisation NumPy, visualiser l'evolution du speedup, illustrer scalabilite multiprocessing, impact du GIL, et gains Haversine (Python / NumPy / Numba).


In [1]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

ROOT = Path("..").resolve()

import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.optim.benchmarks import benchmark_function, compute_features_naive, generate_taxi_df
from src.optim.vectorize import compute_features_vectorized
from src.optim.jit_kernels import haversine_numpy, haversine_numba, warmup_numba, NUMBA_AVAILABLE


def style_plotly(fig):
    fig.update_layout(title_font_size=16, template="plotly_dark")
    return fig


## Benchmark 2.1 : naif vs vectorise

`benchmark_function` sur 50k lignes ; barres comparatives (ms) et speedup.

In [2]:
N = 50_000
df_b = generate_taxi_df(N)

st_naive = benchmark_function(compute_features_naive, df_b, "naif (iterrows)", n_runs=3)
st_vec = benchmark_function(compute_features_vectorized, df_b, "vectorise NumPy", n_runs=3)

t_naive, t_vec = st_naive["mean_ms"], st_vec["mean_ms"]
speedup = t_naive / t_vec if t_vec > 0 else float("nan")

fig = go.Figure(
    data=[
        go.Bar(name="Naif", x=["Temps (ms)"], y=[t_naive], marker_color="indianred"),
        go.Bar(name="Vectorise", x=["Temps (ms)"], y=[t_vec], marker_color="seagreen"),
    ]
)
fig.update_layout(
    title="Speedup Vectorisation NumPy vs iterrows()",
    barmode="group",
    annotations=[
        dict(
            x=0,
            y=max(t_naive, t_vec) * 1.05,
            text=f"Speedup ~ x{speedup:.0f}",
            showarrow=False,
            font=dict(size=14, color="white"),
        )
    ],
)
style_plotly(fig)
fig.show()


┌─────────────────────────────────────────────────┐
│ Benchmark : naif (iterrows)
│ Lignes    : 50,000
│ Runs      : 3
│ Min       : 5839.6 ms
│ Max       : 6812.6 ms
│ Moyenne   : 6434.7 ms
│ Débit     : 7,770 lignes/s
└─────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────┐
│ Benchmark : vectorise NumPy
│ Lignes    : 50,000
│ Runs      : 3
│ Min       : 12.0 ms
│ Max       : 15.8 ms
│ Moyenne   : 13.6 ms
│ Débit     : 3,682,807 lignes/s
└─────────────────────────────────────────────────┘


C:\Users\kevin\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## Courbe speedup vs taille

Pour limiter le temps d'execution, le benchmark **naif** n'est lance que pour `n <= 10_000` ; au-dela, seul le temps vectorise est mesure (speedup en NaN au-dela).

In [3]:
sizes = [1_000, 5_000, 10_000, 50_000, 100_000]
t_naive_list, t_vec_list, speedups = [], [], []

for n in sizes:
    d = generate_taxi_df(n)
    tv = benchmark_function(compute_features_vectorized, d, f"vect n={n}", n_runs=3)["mean_ms"]
    t_vec_list.append(tv)
    if n <= 10_000:
        tn = benchmark_function(compute_features_naive, d, f"naif n={n}", n_runs=3)["mean_ms"]
        t_naive_list.append(tn)
        speedups.append(tn / tv if tv > 0 else np.nan)
    else:
        t_naive_list.append(np.nan)
        speedups.append(np.nan)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=sizes, y=t_naive_list, name="Naif (ms)", mode="lines+markers", line=dict(color="indianred")),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=sizes, y=t_vec_list, name="Vectorise (ms)", mode="lines+markers", line=dict(color="seagreen")),
    secondary_y=False,
)
fig.add_trace(go.Scatter(x=sizes, y=speedups, name="Speedup", mode="lines+markers", line=dict(color="gold")), secondary_y=True)
fig.update_xaxes(type="log", title="n (log)")
fig.update_yaxes(title_text="Temps (ms)", secondary_y=False)
fig.update_yaxes(title_text="Speedup", secondary_y=True)
fig.update_layout(title="Evolution du speedup selon la taille du dataset")
style_plotly(fig)
fig.show()


┌─────────────────────────────────────────────────┐
│ Benchmark : vect n=1000
│ Lignes    : 1,000
│ Runs      : 3
│ Min       : 3.3 ms
│ Max       : 4.9 ms
│ Moyenne   : 4.0 ms
│ Débit     : 252,936 lignes/s
└─────────────────────────────────────────────────┘


┌─────────────────────────────────────────────────┐
│ Benchmark : naif n=1000
│ Lignes    : 1,000
│ Runs      : 3
│ Min       : 108.9 ms
│ Max       : 174.4 ms
│ Moyenne   : 132.1 ms
│ Débit     : 7,571 lignes/s
└─────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────┐
│ Benchmark : vect n=5000
│ Lignes    : 5,000
│ Runs      : 3
│ Min       : 3.8 ms
│ Max       : 5.5 ms
│ Moyenne   : 4.5 ms
│ Débit     : 1,109,500 lignes/s
└─────────────────────────────────────────────────┘


┌─────────────────────────────────────────────────┐
│ Benchmark : naif n=5000
│ Lignes    : 5,000
│ Runs      : 3
│ Min       : 758.2 ms
│ Max       : 970.2 ms
│ Moyenne   : 829.8 ms
│ Débit     : 6,026 lignes/s
└─────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────┐
│ Benchmark : vect n=10000
│ Lignes    : 10,000
│ Runs      : 3
│ Min       : 3.8 ms
│ Max       : 6.4 ms
│ Moyenne   : 4.7 ms
│ Débit     : 2,109,274 lignes/s
└─────────────────────────────────────────────────┘


┌─────────────────────────────────────────────────┐
│ Benchmark : naif n=10000
│ Lignes    : 10,000
│ Runs      : 3
│ Min       : 1142.5 ms
│ Max       : 1256.5 ms
│ Moyenne   : 1218.4 ms
│ Débit     : 8,207 lignes/s
└─────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────┐
│ Benchmark : vect n=50000
│ Lignes    : 50,000
│ Runs      : 3
│ Min       : 12.0 ms
│ Max       : 13.3 ms
│ Moyenne   : 12.8 ms
│ Débit     : 3,900,349 lignes/s
└─────────────────────────────────────────────────┘


┌─────────────────────────────────────────────────┐
│ Benchmark : vect n=100000
│ Lignes    : 100,000
│ Runs      : 3
│ Min       : 17.6 ms
│ Max       : 21.5 ms
│ Moyenne   : 19.4 ms
│ Débit     : 5,155,507 lignes/s
└─────────────────────────────────────────────────┘


## Scalabilité multiprocessing

Mesure sur 4 mois synthétiques de 2 000 lignes, workers limités au nombre de CPU de la machine. `persist_outputs=False` pour ne pas réécrire `data/processed/`.


In [4]:
from src.optim.parallel import process_months_parallel
import os

cpu = os.cpu_count() or 1
worker_counts = sorted({w for w in [1, 2, 4, cpu] if w <= cpu})
rows = []
for n_workers in worker_counts:
    _, elapsed_s = process_months_parallel(
        [1, 2, 3, 4], 2_000, n_workers, persist_outputs=False, use_real=False
    )
    rows.append({"n_workers": n_workers, "elapsed_s": elapsed_s})
curve = pd.DataFrame(rows)
ref = float(curve.loc[curve["n_workers"] == 1, "elapsed_s"].iloc[0])
curve["speedup"] = ref / curve["elapsed_s"]
display(curve)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=curve["n_workers"],
        y=curve["n_workers"],
        mode="lines",
        name="idéal",
        line=dict(dash="dash", color="gray"),
    )
)
fig.add_trace(
    go.Scatter(
        x=curve["n_workers"],
        y=curve["speedup"],
        mode="lines+markers",
        name="mesuré",
        line=dict(color="#38bdf8"),
    )
)
fig.update_layout(
    title="Speedup ProcessPool, 4 mois synthétiques de 2 000 lignes",
    xaxis_title="Workers",
    yaxis_title="Speedup",
)
style_plotly(fig)
fig.show()


[PAR] 4 mois | 1 workers | 4 CPU


[PAR] Mois 1 terminé (51ms) — PID 1744
[PAR] Mois 2 terminé (16ms) — PID 1744
[PAR] Mois 3 terminé (15ms) — PID 1744
[PAR] Mois 4 terminé (19ms) — PID 1744
[PAR] 4 mois | 2 workers | 4 CPU


[PAR] Mois 1 terminé (61ms) — PID 16780
[PAR] Mois 2 terminé (63ms) — PID 15060
[PAR] Mois 3 terminé (42ms) — PID 16780
[PAR] Mois 4 terminé (41ms) — PID 15060


[PAR] 4 mois | 4 workers | 4 CPU


[PAR] Mois 2 terminé (118ms) — PID 19060
[PAR] Mois 1 terminé (123ms) — PID 16752


[PAR] Mois 4 terminé (86ms) — PID 13000
[PAR] Mois 3 terminé (65ms) — PID 3868


,n_workers,elapsed_s,speedup
0,1,2.249434,1.000000
1,2,2.737792,0.821623
2,4,5.167044,0.435342


## GIL

`benchmark_gil_cpu_bound` sur 4 workers et 200 000 itérations par tâche. Le temps processus inclut le démarrage Windows (`spawn`).


In [5]:
from src.optim.parallel import benchmark_gil_cpu_bound

gil = benchmark_gil_cpu_bound(n_workers=4, size=200_000)
labels = ["Séquentiel", "Threading", "Processus"]
vals = [gil["t_seq"], gil["t_threading"], gil["t_multiprocessing"]]
fig = go.Figure(go.Bar(x=labels, y=vals, marker_color=["#888", "#ff7f0e", "#2ca02c"]))
fig.update_layout(
    title="GIL, tâche CPU Python pur (secondes)",
    yaxis_title="Temps (s)",
)
style_plotly(fig)
fig.show()
print(
    f"speedup threading {gil['speedup_threading']:.3f}, "
    f"speedup processus {gil['speedup_processes']:.3f}"
)


╔══════════════════════════════════════════════════════════════╗
║         GIL — TÂCHE CPU-BOUND (Python pur)                   ║
║         4 workers | 200,000 itérations/tâche                 ║
╠══════════════╦══════════════╦═════════════╦════════════════╣
║   Mode       ║   Temps (s)  ║   Speedup   ║  Commentaire   ║
╠══════════════╬══════════════╬═════════════╬════════════════╣
║ Séquentiel   ║     0.42s    ║    1.00x    ║  référence     ║
║ Threading    ║     0.56s    ║    0.75x    ║  GIL bloque !  ║
║ Processes    ║     0.80s    ║    0.52x    ║  vrai parallél ║
╚══════════════╩══════════════╩═════════════╩════════════════╝


speedup threading 0.751, speedup processus 0.522


## Haversine : Python vers NumPy vers Numba

Barres en **echelle log** ; speedup relatif a une boucle Python scalaire (mesure sur sous-echantillon pour Python pur).

In [6]:
import time
from src.optim.jit_kernels import haversine_python

n = 200_000
lat1 = np.random.uniform(40.6, 40.9, n)
lon1 = np.random.uniform(-74.1, -73.9, n)
lat2 = lat1 + np.random.normal(0, 0.01, n)
lon2 = lon1 + np.random.normal(0, 0.01, n)

m = 2_000
t0 = time.perf_counter()
_ = [haversine_python(float(lat1[i]), float(lon1[i]), float(lat2[i]), float(lon2[i])) for i in range(m)]
t_py_s = (time.perf_counter() - t0) / m * n

t0 = time.perf_counter()
_ = haversine_numpy(lat1, lon1, lat2, lon2)
t_np_ms = (time.perf_counter() - t0) * 1000

labels = ["Python (extrap.)", "NumPy"]
vals_ms = [max(t_py_s * 1000, 1e-6), max(t_np_ms, 1e-6)]
if NUMBA_AVAILABLE:
    warmup_numba()
    t0 = time.perf_counter()
    _ = haversine_numba(lat1, lon1, lat2, lon2)
    t_nb_ms = (time.perf_counter() - t0) * 1000
    labels.append("Numba")
    vals_ms.append(max(t_nb_ms, 1e-6))

base = vals_ms[0]
speeds = [base / v for v in vals_ms]

fig = go.Figure(go.Bar(x=labels, y=speeds, marker_color=["#d62728", "#1f77b4", "#9467bd"][: len(labels)]))
fig.update_layout(
    title="Speedup Haversine : Python -> NumPy -> Numba",
    yaxis_type="log",
    yaxis_title="Speedup relatif (log)",
)
style_plotly(fig)
fig.show()


[OK] Warmup Numba terminé


## Lecture

Les speedups affichés au-dessus sont ceux mesurés dans ce notebook (jeu synthétique, machine à 4 CPU). Les cibles de l'énoncé (×100, ×4, ×200) sont des objectifs, pas les valeurs de ces graphiques. Le fichier `reports/benchmarks.json` reprend une autre passe, avec sa commande.
